In [1]:
import pandas as pd
import numpy as np
import os, sys
from tqdm.notebook import tqdm
from pathlib import Path
from plotly import express as px
import plotly.graph_objects as go
from sklearn.linear_model import  LinearRegression

In [2]:
file_path = Path("../rogii-wellbore-geology-prediction/train",)
well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in file_path.glob("*__horizontal_well.csv")
            }

In [3]:
well_data = pd.read_csv(well_files["059c8f24"]["Well"])
pred_start = well_data.TVT_input.shift(1).dropna().index[-1]

In [4]:
wells_data = pd.concat([pd.read_csv(d["Well"]) for d in tqdm(well_files.values())],axis=0)

  0%|          | 0/773 [00:00<?, ?it/s]

In [5]:
wells_data = wells_data.dropna(subset=["X", "Y","Z",'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',])

In [10]:
X = wells_data.loc[:, ["X", "Y"]]
y = wells_data.loc[:,['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',]]
formation_model = LinearRegression().fit(X=X,y=y)

In [12]:
formations = pd.DataFrame(formation_model.predict(well_data[["X","Y"]]), index=well_data.index, columns=['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA',])

In [13]:
well_data.loc[pred_start:,"Z"]

1740   -8351.51
1741   -8351.56
1742   -8351.60
1743   -8351.64
1744   -8351.69
         ...   
7993   -8574.32
7994   -8574.27
7995   -8574.23
7996   -8574.18
7997   -8574.14
Name: Z, Length: 6258, dtype: float64

In [6]:
def plot_wellbore(well_data):
    pred_start = well_data.TVT_input.shift(1).dropna().index[-1]
    fig = px.scatter_3d(well_data, x="X", y="Y", z="Z", color="TVT").update_layout(height=800, width=800)
    fig.update_traces(marker=dict(size=2))
    start_point = well_data.loc[pred_start]
    fig.add_trace(go.Scatter3d(
        x=[start_point["X"]],
        y=[start_point["Y"]],
        z=[start_point["Z"]],
        mode="markers",
        marker=dict(color="red", size=8),
        name="Prediction Start",
        showlegend=False
    ))
    fig.show()